# Práctica 5 · Modelos de decisión

## 1 · Preparar
En Colab, selecciona **T4 GPU** si está disponible; también puedes usar CPU. PyTorch ya viene instalado.

Si ejecutaste la preparación anterior, usa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** antes de empezar de nuevo. Si ya has importado Transformers o Hugging Face Hub, reinicia la sesión después de instalar y continúa con la siguiente celda.

Estas versiones mantienen Hugging Face Hub en la rama 1.x, compatible con los requisitos de Gradio y Diffusers indicados por Colab.


In [ ]:
%pip -q install "laya==0.3.20" "transformers==5.20.2" "huggingface-hub==1.32.0" pandas matplotlib

In [ ]:
import os
os.environ["USE_TF"] = "0"  # Evita que Transformers cargue TensorFlow en Colab.
os.environ["USE_FLAX"] = "0"

import copy
import html
import json
import platform
import time
from importlib.metadata import version
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.generation.streamers import BaseStreamer
import laya

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
LAYA_ID = "convaiinnovations/laya"
LAYA_REV = "7b928d828b7b0e022f929d9bd2e44165aa270148"
LLM_ID = "Qwen/Qwen2.5-0.5B-Instruct"
LLM_REV = "7ae557604adf67be50417f59c2c2f167def9a775"
MAX_NEW_TOKENS = 48

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

environment = {
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": version("transformers"), "laya": version("laya"),
    "huggingface-hub": version("huggingface-hub"),
    "hardware": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "laya_revision": LAYA_REV, "llm_revision": LLM_REV,
}
display(pd.Series(environment, name="Entorno real"))

In [ ]:
# Solo se descarga la variante multilingüe, sin kernels ni compilación adicionales.
# El SDK 0.3.20 no recibe revision: descargamos antes un snapshot fijo.
laya_path = snapshot_download(
    LAYA_ID, revision=LAYA_REV,
    allow_patterns=["multilingual/rl_agent_config.json", "multilingual/model.safetensors",
                    "multilingual/tokenizer/*", "multilingual/encoder/*"],
)
agent = laya.load(str(Path(laya_path) / "multilingual"), device=DEVICE, fast=False)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, revision=LLM_REV)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID, revision=LLM_REV, dtype=DTYPE, use_safetensors=True,
).to(DEVICE).eval()
print("Modelos cargados. Descarga y carga quedan fuera de los tiempos de inferencia.")

## 2 · Mirar dentro de la configuración
Laya puntúa opciones definidas en cada petición. Qwen genera tokens con una plantilla de chat. Mira parámetros, contexto, precisión y límites antes de preguntar.

In [ ]:
display(pd.DataFrame([
    {"modelo": "Laya multilingüe", "parámetros M": sum(p.numel() for p in agent.model.parameters()) / 1e6,
     "dispositivo": str(agent.device), "pesos": str(next(agent.model.parameters()).dtype),
     "cálculo": str(agent.dtype), "límite de entrada configurado": agent.cfg.get("max_len"),
     "presupuesto de pregunta/opciones": agent.cfg.get("head_max_len")},
    {"modelo": "Qwen 0.5B", "parámetros M": sum(p.numel() for p in llm.parameters()) / 1e6,
     "dispositivo": str(llm.device), "pesos": str(next(llm.parameters()).dtype),
     "cálculo": str(DTYPE), "límite de entrada configurado": llm.config.max_position_embeddings,
     "presupuesto de pregunta/opciones": None},
]).round(1))
print("Configuración completa de Laya:")
print(json.dumps(agent.cfg, indent=2, ensure_ascii=False))
print("Qwen:", llm.config.model_type, "· capas:", llm.config.num_hidden_layers,
      "· generación greedy · máximo", MAX_NEW_TOKENS, "tokens nuevos")

## 3 · Construir una petición
**Estado + pregunta + criterios.** Las instrucciones dicen qué decidir; los criterios definen las salidas posibles. No se pide una explicación libre.

Tres tipos: `choice` elige una categoría; `noul` devuelve P(sí); `score` devuelve una puntuación esperada sobre niveles ordenados (0, 1, 2…). No son temperaturas ni límites de tokens de generación.

In [ ]:
# Estado simulado: edita este texto y vuelve a ejecutar las siguientes celdas.
STATE = "Me habéis cobrado dos veces este mes. Quiero que me devolváis el cargo duplicado."
QUESTIONS = {
    "departamento": {
        "type": "choice",
        "instructions": "¿Qué equipo debe atender este mensaje?",
        "criteria": {
            "facturacion": "Cobros, facturas y devoluciones de dinero.",
            "tecnico": "Errores de la aplicación, fallos de acceso y averías.",
            "ventas": "Precios, planes y contratación.",
            "otros": "Ninguno de los anteriores.",
        },
    },
    "reembolso": {
        "type": "noul",
        "instructions": "¿Solicita explícitamente una devolución de dinero?",
    },
    "urgencia": {
        "type": "score",
        "instructions": "Valora la urgencia indicada en el mensaje.",
        "criteria": ["Sin urgencia explícita", "Necesita atención pronto", "Servicio bloqueado o plazo inmediato"],
    },
}
# Texto y JSON como estado; una pregunta, varias preguntas o varios estados.
result = agent.predict(STATE, QUESTIONS)
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
answers = result["answers"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key in zip(axes[:2], ["departamento", "urgencia"]):
    probs = answers[key]["probabilities"]
    ax.bar(list(probs), list(probs.values()), color="#536cb3")
    ax.set_ylim(0, 1)
    ax.set_title(key.capitalize(), fontsize=18)
    ax.tick_params(axis="x", labelrotation=20)
    ax.set_ylabel("Probabilidad del modelo")
p_yes = answers["reembolso"]["noul"]
axes[2].bar(["No", "Sí"], [1 - p_yes, p_yes], color=["#aab2c6", "#536cb3"])
axes[2].set_ylim(0, 1)
axes[2].set_title("¿Pide reembolso?", fontsize=18)
fig.suptitle("Predicciones reales · mensaje simulado", fontsize=20)
plt.tight_layout()
plt.show()
print("Urgencia esperada:", answers["urgencia"]["score"], "/ 2; no es P(urgente).")

Las probabilidades pueden estar equivocadas o ser demasiado confiadas. Este ejemplo no valida su calibración. `confidence` tampoco significa lo mismo en todos los tipos del SDK: inspecciona `probabilities` y `answer_confidence`. No uses `action.act_probability` para automatizar esta práctica.

In [ ]:
ONE_QUESTION = {"departamento": QUESTIONS["departamento"]}
json_state = {"asunto": "Cargo duplicado", "mensaje": STATE}  # Datos simulados.
json_result = agent.predict(json_state, ONE_QUESTION)
batch_results = agent.predict_batch([
    STATE, "No puedo iniciar sesión: aparece un error al abrir la aplicación."
], ONE_QUESTION)
display(pd.DataFrame([
    {"petición": "JSON · una pregunta", "salida": json_result["answers"]["departamento"]["choice"]},
    *[{"petición": f"Lote · estado {i+1}", "salida": r["answers"]["departamento"]["choice"]}
      for i, r in enumerate(batch_results)],
]))

## 4 · ¿Qué cambia al cambiar el prompt?
Mismo mensaje, otra política de enrutamiento. Compara distribuciones: el cambio de instrucciones puede cambiar el resultado, pero no lo garantiza. Prueba después un mensaje ambiguo.

In [ ]:
policy = copy.deepcopy(ONE_QUESTION)
policy["departamento"]["instructions"] = (
    "¿Qué equipo debe atender este mensaje? Política del taller: "
    "las solicitudes de devolución se derivan a otros para revisión manual."
)
policy["departamento"]["criteria"]["facturacion"] = "Consultas de cobros y facturas sin solicitud de devolución."
policy["departamento"]["criteria"]["otros"] = "Devoluciones para revisión manual u otros asuntos."
policies = {"Habitual": ONE_QUESTION, "Revisión de devoluciones": policy}
policy_probs = {
    name: agent.predict(STATE, questions)["answers"]["departamento"]["probabilities"]
    for name, questions in policies.items()
}
pd.DataFrame(policy_probs).plot.bar(figsize=(10, 4), ylim=(0, 1), rot=0,
                                   ylabel="Probabilidad del modelo", title="Mismo estado · distintos criterios")
plt.tight_layout()
plt.show()

## 5 · Misma tarea: elegir departamento
Ambos reciben el mismo mensaje y los mismos criterios. A Qwen le pedimos **solo** `{"departamento": "una_clave"}`, sin explicaciones. Así no alargamos artificialmente la salida generativa.

Laya devuelve una distribución completa. El JSON de Qwen ofrece una elección, no una probabilidad calibrada. Las dos implementaciones tienen distinto tamaño y entrenamiento: la diferencia medida no se puede atribuir solo a su arquitectura.

In [ ]:
def llm_messages(state):
    task = json.dumps({"state": state, "questions": ONE_QUESTION}, ensure_ascii=False)
    return [
        {"role": "system", "content": 'Clasifica usando la pregunta y sus criterios. Devuelve solo JSON válido con una única clave "departamento" y como valor una clave de los criterios. Sin explicación.'},
        {"role": "user", "content": task},
    ]

print("Petición de decisiones:")
print(json.dumps({"state": STATE, "questions": ONE_QUESTION}, ensure_ascii=False, indent=2))
print("\nPrompt de chat equivalente:")
print(tokenizer.apply_chat_template(llm_messages(STATE), tokenize=False, add_generation_prompt=True))

In [ ]:
class TokenTrace(BaseStreamer):
    """Recoge cada token de generate, no fragmentos simulados ni retardos añadidos."""
    def __init__(self, started, on_update=None):
        self.started, self.on_update = started, on_update
        self.skip_prompt = True
        self.ids, self.times_ms = [], []

    def put(self, value):
        if self.skip_prompt:
            self.skip_prompt = False
            return
        for token_id in value.reshape(-1).tolist():
            self.ids.append(token_id)
            self.times_ms.append((time.perf_counter() - self.started) * 1000)
        if self.on_update:
            self.on_update(tokenizer.decode(self.ids, skip_special_tokens=True), len(self.ids))

    def end(self):
        pass

@torch.inference_mode()
def run_llm(state, on_update=None):
    sync()
    started = time.perf_counter()
    inputs = tokenizer.apply_chat_template(llm_messages(state), tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors="pt").to(DEVICE)
    trace = TokenTrace(started, on_update)
    output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id, streamer=trace)
    sync()
    new_ids = output[0, inputs["input_ids"].shape[-1]:].tolist()
    raw = tokenizer.decode(new_ids, skip_special_tokens=True)
    try:
        parsed = json.loads(raw)
        choice = parsed.get("departamento") if isinstance(parsed, dict) else None
        valid = isinstance(parsed, dict) and set(parsed) == {"departamento"} and choice in ONE_QUESTION["departamento"]["criteria"]
    except (ValueError, TypeError):
        choice, valid = None, False
    elapsed_ms = (time.perf_counter() - started) * 1000
    return {"model": "Qwen", "ms": elapsed_ms, "ttft_ms": trace.times_ms[0] if trace.times_ms else None,
            "times_ms": trace.times_ms, "tokens": len(new_ids), "raw": raw,
            "choice": choice, "valid": valid, "input_tokens": inputs["input_ids"].shape[-1],
            "hit_limit": len(new_ids) >= MAX_NEW_TOKENS}

def run_decision(state):
    sync()
    started = time.perf_counter()
    response = agent.predict(state, ONE_QUESTION)
    sync()
    elapsed_ms = (time.perf_counter() - started) * 1000
    answer = response["answers"]["departamento"]
    return {"model": "Laya", "ms": elapsed_ms, "choice": answer["choice"],
            "valid": answer["choice"] in ONE_QUESTION["departamento"]["criteria"],
            "tokens": 0, "response": response, "input_tokens": response["usage"]["input_tokens"]}

# Calentamiento: misma tarea. No se incluye en las mediciones.
_ = run_decision(STATE)
_ = run_llm(STATE)
assert next(agent.model.parameters()).device == llm.device, "Los dos modelos deben seguir en el mismo dispositivo para comparar."
print("Calentamiento terminado. Dispositivo:", llm.device)

### Ver cómo llega la salida
Ejecuta para ver una devolución completa de Laya y el texto de Qwen mientras se genera. Se ejecutan por separado, cada uno con su propio reloj. La interfaz introduce sobrecoste: los tiempos de esta demostración no son el ensayo de rendimiento.

In [ ]:
live_decision = run_decision(STATE)

def panel(text, count):
    return HTML(
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;font:18px system-ui">'
        '<div style="border:1px solid #bbb;padding:20px"><b>Laya · respuesta completa</b>'
        f'<h2>{html.escape(live_decision["choice"])}</h2>'
        f'<p>{live_decision["ms"]:.1f} ms · 0 tokens generados</p></div>'
        '<div style="border:1px solid #bbb;padding:20px"><b>Qwen · streaming real</b>'
        f'<pre style="white-space:pre-wrap">{html.escape(text)}</pre>'
        f'<p>{count} tokens recibidos</p></div></div>'
    )

view = display(panel("Esperando…", 0), display_id=True)
def update_live(text, count):
    view.update(panel(text, count))
live_llm = run_llm(STATE, on_update=update_live)
print(f'Qwen: primer token {live_llm["ttft_ms"]:.1f} ms · total con interfaz {live_llm["ms"]:.1f} ms')
print("JSON válido:", live_llm["valid"], "· límite de tokens alcanzado:", live_llm["hit_limit"])

## 6 · Medir sin actualizar la interfaz
Tres repeticiones después de calentar; alternamos el orden. Mismo estado, una pregunta y el mismo dispositivo. Tiempo de extremo a extremo local: preparación/tokenización, inferencia y obtención de salida; excluye descarga, carga y dibujo.

La curva cuenta tokens recibidos por el streamer, incluido EOS si llega. El primer token puede no ser texto visible. Cada muestra refleja una llamada local secuencial; no hay red ni concurrencia. Los tokens de entrada no son comparables entre tokenizadores y Laya prepara una secuencia por pregunta.

In [ ]:
runs, errors = [], []
for repetition in range(3):
    order = [run_decision, run_llm] if repetition % 2 == 0 else [run_llm, run_decision]
    for runner in order:
        try:
            measurement = runner(STATE)
            if next(agent.model.parameters()).device != llm.device:
                raise RuntimeError("Dispositivos distintos después de un fallback: no comparar estas medidas.")
            runs.append({**measurement, "repetition": repetition + 1})
        except Exception as exc:
            errors.append({"repetition": repetition + 1, "runner": runner.__name__, "error": str(exc)})

frame = pd.DataFrame([{k: r.get(k) for k in ["model", "repetition", "ms", "ttft_ms", "tokens", "choice", "valid", "hit_limit"]} for r in runs])
display(frame.round(2))
if errors:
    display(pd.DataFrame(errors))
if not runs:
    raise RuntimeError("No hay medidas válidas. Revisa los errores antes de continuar.")
summary = frame.groupby("model")["ms"].agg(["median", "min", "max", "count"])
display(summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, group in frame.groupby("model"):
    axes[0].scatter([name] * len(group), group["ms"], s=65, label=name)
axes[0].set_ylabel("Tiempo hasta respuesta completa · ms")
axes[0].set_title("Medidas reales · cada punto es una ejecución")
for run in runs:
    if run["model"] == "Qwen":
        axes[1].step([0] + run["times_ms"], [0] + list(range(1, len(run["times_ms"]) + 1)),
                     where="post", alpha=.6, label=f'Qwen · {run["repetition"]}')
    else:
        axes[1].axvline(run["ms"], color="#b76b37", alpha=.5, linestyle="--",
                       label=f'Laya · respuesta {run["repetition"]}')
axes[1].set_xlabel("Tiempo desde el inicio de cada llamada · ms")
axes[1].set_ylabel("Tokens generados por Qwen")
axes[1].set_title("Generación progresiva / decisión completa")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7 · ¿Decide bien o solo rápido?
Cuatro mensajes **simulados y etiquetados a mano**. Mira los errores de clasificación, el JSON inválido y el posible truncamiento. No constituyen una evaluación representativa ni un ranking de modelos.

In [ ]:
CASES = [
    (STATE, "facturacion"),
    ("La aplicación muestra un error y no puedo entrar en mi cuenta.", "tecnico"),
    ("¿Cuánto cuesta contratar vuestro plan de empresa?", "ventas"),
    ("Gracias por vuestra ayuda, solo quería saludar.", "otros"),
]
quality = []
for state, expected in CASES:
    for runner in [run_decision, run_llm]:
        try:
            r = runner(state)
            quality.append({"mensaje simulado": state, "esperado": expected, "modelo": r["model"],
                            "salida": r["choice"], "formato válido": r["valid"],
                            "acierto": bool(r["valid"] and r["choice"] == expected),
                            "texto original": r.get("raw"), "límite alcanzado": r.get("hit_limit", False)})
        except Exception as exc:
            quality.append({"mensaje simulado": state, "esperado": expected,
                            "modelo": runner.__name__, "error": str(exc), "acierto": False})
display(pd.DataFrame(quality))

## 8 · Dónde probarlo

| Caso sencillo | Pregunta tipada | Uso de la salida |
|---|---|---|
| Enrutar soporte | `choice`: facturación / técnico / ventas / otros | Elegir una cola |
| Elegir herramienta | `choice`: buscar documento / consultar pedido / ninguna | Proponer el siguiente paso |
| Detectar una solicitud | `noul`: ¿pide reembolso? | Marcar para revisión |
| Priorizar tickets | `score`: sin urgencia / pronto / bloqueo | Ordenar una bandeja |

Ejercicio: añade la opción «revisión» y prueba mensajes ambiguos. Repite `noul` como `choice` con dos opciones y compara: el modelo también puede ser sensible a las etiquetas. Ajusta cualquier umbral con ejemplos etiquetados propios; un porcentaje alto no garantiza un acierto.

Para escribir una respuesta libre o explicar un razonamiento, conserva el LLM. Para decidir entre opciones pequeñas y repetidas, prueba el modelo de decisión y valida su calidad. En esta práctica ninguna decisión ejecuta acciones externas.

In [ ]:
# Exportación sin salidas precargadas: entorno, configuración, tiempos y errores reales.
report = {"environment": environment, "state_simulated": STATE, "questions": ONE_QUESTION,
          "laya_config": agent.cfg, "max_new_tokens": MAX_NEW_TOKENS,
          "runs": runs, "errors": errors, "quality_examples_simulated": quality}
Path("decisiones-resultados.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str))
frame.to_csv("decisiones-tiempos.csv", index=False)
print("Guardados decisiones-resultados.json y decisiones-tiempos.csv. Descárgalos desde el panel Archivos de Colab.")

<small>Fuentes: [Laya · pesos y límites](https://huggingface.co/convaiinnovations/laya) · [SDK 0.3.20](https://pypi.org/project/laya/0.3.20/) · [Código del SDK](https://github.com/NandhaKishorM/laya) · [Qwen · ficha](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) · [TypeSafe · Jev](https://typesafe.ai/). La calibración, calidad y tiempos deben comprobarse en la tarea y el hardware propios.</small>